# Oasis 500M — ORE24 Colab 실행

**런타임 → 런타임 유형 변경 → GPU**를 선택한 후 위에서부터 실행하세요. 먼저 8프레임 영상으로 확인합니다. 실시간 게임이 아닌 영상 생성 예제입니다.

모델 다운로드 전 [Etched/oasis-500m](https://huggingface.co/Etched/oasis-500m)에서 본인 계정으로 접근 조건을 확인하고 동의하세요. 조건에는 연락처 정보 공유가 포함됩니다. GitHub ORE24와 Hugging Face 계정은 별개입니다.

Colab 왼쪽 열쇠 아이콘(Secrets)에 `HF_TOKEN`을 등록하고 이 노트북의 접근을 허용하세요. 토큰은 [Hugging Face 설정](https://huggingface.co/settings/tokens)에서 해당 모델을 읽을 수 있는 권한으로 생성합니다. 코드 셀이나 출력에 토큰을 붙여 넣지 마세요.

GPU 종류·사용 가능 시간은 Colab 상황에 따라 달라집니다. 세션이 끝나기 전에 결과 영상을 내려받으세요.


## 1. GPU와 환경 확인

In [ ]:
import subprocess, sys
import torch
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "런타임 → 런타임 유형 변경 → GPU 선택 후 다시 실행하세요."
print("GPU:", torch.cuda.get_device_name(0))
print("GPU 메모리:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GiB")
subprocess.run(["nvidia-smi"], check=True)


## 2. 포크 복제 및 의존성 설치
Colab에 설치된 CUDA용 PyTorch는 유지합니다. 소스는 확인한 커밋으로 고정합니다.

In [ ]:
from pathlib import Path
REPO_DIR = Path("/content/open-oasis")
SOURCE_REVISION = "f59deef2c019c212bd0c5a3a5b986a51f3701847"
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/ORE24/open-oasis.git", str(REPO_DIR)], check=True)
subprocess.run(["git", "checkout", "--detach", SOURCE_REVISION], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "einops>=0.8,<0.9", "timm>=1.0,<1.1", "av>=14,<17", "safetensors>=0.5,<1", "huggingface_hub>=0.34,<2", "tqdm", "pillow"], check=True)


## 3. 영상 저장 호환성 설정
최신 torchvision에서 제거된 영상 입출력 API 대신 PyAV를 사용합니다. Colab 작업 복사본의 영상 입출력만 변경하며 모델 계산은 그대로 유지합니다.

In [ ]:
# 再実行しても同じ置換結果になる設定です。
VIDEO_IO_SOURCE = 'import av\nimport numpy as np\nimport torch\n\ndef read_video(filename, pts_unit="sec"):\n    with av.open(str(filename)) as container:\n        stream = container.streams.video[0]\n        rate = float(stream.average_rate or 20)\n        frames = [frame.to_ndarray(format="rgb24") for frame in container.decode(video=0)]\n    return torch.from_numpy(np.stack(frames)), torch.empty((1, 0)), {"video_fps": rate}\n\ndef write_video(filename, video_array, fps):\n    frames = video_array.detach().cpu().numpy()\n    with av.open(str(filename), mode="w") as container:\n        stream = container.add_stream("libx264", rate=int(fps))\n        stream.width = frames.shape[2]\n        stream.height = frames.shape[1]\n        stream.pix_fmt = "yuv420p"\n        for array in frames:\n            frame = av.VideoFrame.from_ndarray(array, format="rgb24")\n            for packet in stream.encode(frame):\n                container.mux(packet)\n        for packet in stream.encode():\n            container.mux(packet)\n'
(REPO_DIR / "colab_video_io.py").write_text(VIDEO_IO_SOURCE)
replacements = {
    "generate.py": ("from torchvision.io import read_video, write_video", "from colab_video_io import read_video, write_video"),
    "utils.py": ("from torchvision.io import read_image, read_video", "from torchvision.io import read_image\nfrom colab_video_io import read_video"),
}
for filename, (old, new) in replacements.items():
    path = REPO_DIR / filename
    source = path.read_text()
    if old not in source and new not in source:
        raise RuntimeError(f"소스 구조가 변경되었습니다: {filename}")
    path.write_text(source.replace(old, new))
subprocess.run([sys.executable, "-c", "import generate; print('모델 모듈 로드 성공')"], cwd=REPO_DIR, check=True)


## 4. Hugging Face 인증 및 모델 다운로드
접근 승인을 받은 계정의 `HF_TOKEN` Secret을 사용합니다. 401/403 오류가 나면 모델 접근 승인과 토큰 권한을 확인하세요. 로그인 정보를 GitHub에 저장하지 않습니다.

In [ ]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import GatedRepoError, HfHubHTTPError
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    raise RuntimeError("Colab Secrets에 HF_TOKEN을 등록하고 노트북 접근을 허용하세요.") from None
try:
    checkpoint_paths = {
        name: hf_hub_download(repo_id="Etched/oasis-500m", filename=name, token=hf_token)
        for name in ("oasis500m.safetensors", "vit-l-20.safetensors")
    }
except (GatedRepoError, HfHubHTTPError):
    raise RuntimeError("모델 다운로드 실패: Hugging Face 모델 페이지 접근 승인, 토큰 읽기 권한, 네트워크를 확인하세요.") from None
finally:
    del hf_token
print("두 모델 다운로드 완료")


## 5. 짧은 영상 생성
처음에는 8프레임으로 실행하세요. 성공 후 16, 32로 늘릴 수 있습니다. 메모리 부족 시 프레임 수를 줄이세요.

In [ ]:
NUM_FRAMES = 8
DDIM_STEPS = 10
OUTPUT_PATH = REPO_DIR / "oasis_colab.mp4"
assert 2 <= NUM_FRAMES <= 32, "샘플 행동을 사용하는 첫 테스트는 2~32프레임으로 설정하세요."
assert DDIM_STEPS > 0
# 이전 결과가 새 실행의 결과로 표시되지 않도록 제거합니다.
OUTPUT_PATH.unlink(missing_ok=True)
subprocess.run([
    sys.executable, "generate.py",
    "--oasis-ckpt", checkpoint_paths["oasis500m.safetensors"],
    "--vae-ckpt", checkpoint_paths["vit-l-20.safetensors"],
    "--num-frames", str(NUM_FRAMES),
    "--ddim-steps", str(DDIM_STEPS),
    "--output-path", str(OUTPUT_PATH),
], cwd=REPO_DIR, check=True)
assert OUTPUT_PATH.is_file() and OUTPUT_PATH.stat().st_size > 0


## 6. 결과 재생

In [ ]:
from IPython.display import Video, display
assert OUTPUT_PATH.is_file(), "5번 셀에서 영상 생성을 먼저 완료하세요."
display(Video(str(OUTPUT_PATH), embed=True, html_attributes="controls loop"))


## 7. 영상 다운로드
세션이 종료되기 전에 실행하세요.

In [ ]:
from google.colab import files
files.download(str(OUTPUT_PATH))
